# Ring Resonator — Post-Training Analysis

Load a `checkpoint.pt` produced by either `ring_resonator_scaling_batch.py` or
`ring_resonator_arbitrary_batch.py`, reconstruct the simulation, and reproduce
all training plots interactively.

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import torch

from nlse import (
    split_step_fourier_xpm_batch,
    get_hg_basis,
    time_to_hg,
    hg_to_time,
    plot_mode_hg_coeffs,
)

%matplotlib inline

## 1. Load checkpoint

Point `RUN_DIR` at the output directory of a finished training run.

In [ ]:
RUN_DIR = "runs/ring_scaling_small"  # <-- change this

ckpt = torch.load(os.path.join(RUN_DIR, "checkpoint.pt"), map_location="cpu")
args = ckpt["args"]
scenario = ckpt["scenario"]

with open(os.path.join(RUN_DIR, "losses.json")) as f:
    loss_data = json.load(f)
losses = loss_data["losses"]

print(f"Scenario : {scenario}")
print(f"Lz={args['Lz']}, Nz={args['Nz']}, N_resonator={args['N_resonator']}")
print(f"num_modes={args['num_modes']}, loss_fn={args['loss_fn']}")
print(f"Final loss: {loss_data['final_loss']:.6f}")
print(f"Per-mode fidelity: {loss_data['per_mode_fidelity']}")

## 2. Reconstruct simulation

In [ ]:
Lz = args["Lz"]
Nz = args["Nz"]
Lt = args["Lt"]
Nt = args["Nt"]
N_modes = args["N_modes"]
B = args["B"]
beta2_j = args["beta2_j"]
beta2_k = args["beta2_k"]
gamma_j = args["gamma_j"]
gamma_k = args["gamma_k"]
tau = args["tau"]
R = args["R"]
amplitude = args["amplitude"]
N_resonator = args["N_resonator"]
num_modes = args["num_modes"]

dz = Lz / Nz
dt = Lt / Nt
r_coeff = np.sqrt(R)
t_coeff = np.sqrt(1 - R)

t = torch.linspace(-Lt / 2, Lt / 2, Nt)
hg_basis = get_hg_basis(N_modes, t, tau)
hg_basis_B = hg_basis[:B]

weak_input = amplitude * torch.exp(-(t ** 2) / (2 * tau ** 2))
A_j_inputs = hg_basis[:num_modes]

a_2 = np.sqrt(-beta2_k / (gamma_k * tau ** 2))
strong_input = a_2 * torch.cosh(t / tau) ** (-1)

y_hg = torch.zeros(num_modes, B)
for i in range(num_modes):
    y_hg[i, i] = 1.0

## 3. Load trained parameters & run forward pass

In [ ]:
def beam_splitter(a, d, r, tc):
    return tc * a + r * d, r * a + tc * d


if scenario == "scaling":
    A_k_coeffs = ckpt["A_k_coeffs"]
    alphas = ckpt["alphas"]
    A_k_time = hg_to_time(A_k_coeffs, hg_basis)
    A_k_stack = torch.stack([A_k_time * alphas[i] for i in range(N_resonator)])

    def forward():
        A_j = A_j_inputs * t_coeff
        for i in range(N_resonator):
            A_k_scaled = A_k_time * alphas[i]
            A_j_ev, _ = split_step_fourier_xpm_batch(
                A_j, A_k_scaled, dz, Nz, beta2_j, beta2_k, gamma_j, gamma_k, Lt)
            A_j = A_j_ev[:, :, -1]
            A_j, _ = beam_splitter(weak_input, A_j, r_coeff, t_coeff)
        return A_j * t_coeff

elif scenario == "arbitrary":
    A_k_hg_stack = ckpt["A_k_hg_stack"]
    A_k_stack = torch.stack([hg_to_time(A_k_hg_stack[i], hg_basis) for i in range(N_resonator)])

    def forward():
        A_j = A_j_inputs * t_coeff
        for i in range(N_resonator):
            A_k_time_i = hg_to_time(A_k_hg_stack[i], hg_basis)
            A_j_ev, _ = split_step_fourier_xpm_batch(
                A_j, A_k_time_i, dz, Nz, beta2_j, beta2_k, gamma_j, gamma_k, Lt)
            A_j = A_j_ev[:, :, -1]
            A_j, _ = beam_splitter(weak_input, A_j, r_coeff, t_coeff)
        return A_j * t_coeff

else:
    raise ValueError(f"Unknown scenario: {scenario}")

with torch.no_grad():
    A_j_final = forward()
    final_hg = torch.stack([time_to_hg(A_j_final[i], hg_basis_B, dt) for i in range(num_modes)])
    dots = torch.sum(final_hg.conj() * y_hg, dim=1)
    per_mode_fid = torch.abs(dots) ** 2

print("Per-mode fidelity (recomputed):")
for m in range(num_modes):
    print(f"  Mode {m}: {per_mode_fid[m].item():.6f}")
print(f"  Average : {per_mode_fid.mean().item():.6f}")
print(f"  Trace   : {(torch.abs(torch.sum(dots) / num_modes) ** 2).item():.6f}")

## 4. Plots

### 4.1 Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, label="Training Loss", color="tab:blue")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Curve")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

### 4.2 Strong pulse HG coefficients

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
init_coeffs = time_to_hg(strong_input, hg_basis, dt).numpy()
mode_indices = np.arange(N_modes)
ax.stem(mode_indices, np.abs(init_coeffs) ** 2, linefmt="r-", markerfmt="ro",
        basefmt=" ", label="Initial (soliton)")

if scenario == "scaling":
    opt_coeffs = A_k_coeffs.numpy()
    ax.stem(mode_indices, np.abs(opt_coeffs) ** 2, linefmt="b-", markerfmt="bo",
            basefmt=" ", label="Optimized")
else:
    cmap_loops = plt.cm.viridis
    for loop_idx in range(N_resonator):
        opt_coeffs = A_k_hg_stack[loop_idx].numpy()
        color = cmap_loops(loop_idx / N_resonator)
        ax.stem(mode_indices, np.abs(opt_coeffs) ** 2, linefmt="-", markerfmt="o",
                basefmt=" ", label=f"Loop {loop_idx}")
        ax.get_children()[-3].set_color(color)
        ax.get_children()[-4].set_color(color)

ax.set_xlabel("HG Mode Index")
ax.set_ylabel("Coefficient Intensity |c_n|\u00b2")
ax.set_title("Strong Pulse HG Coefficients: Initial vs Optimized")
ax.legend(fontsize=7, ncol=3)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### 4.3 Alphas (scaling only)

In [ ]:
if scenario == "scaling":
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(alphas.numpy(), marker="o", linestyle="-", color="tab:green")
    ax.set_xlabel("Resonator Step")
    ax.set_ylabel("Alpha (Strong Pulse Scale)")
    ax.set_title(r"Optimized $\alpha$ (Strong Pulse Amplitudes)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("(Alphas plot skipped — arbitrary scenario has independent pulses per loop)")

### 4.4 Strong pulse evolution across resonator loops

In [ ]:
fig = plt.figure(figsize=(12, 7))
cmap = plt.cm.plasma
n_pulses = len(A_k_stack)
colors = [cmap(i / n_pulses) for i in range(n_pulses)]
t_np = t.numpy()
for i, pulse in enumerate(A_k_stack):
    intensity = np.abs(pulse.numpy()) ** 2
    plt.plot(t_np, intensity, color=colors[i], alpha=0.7, linewidth=1.5)
    max_idx = np.argmax(intensity)
    plt.text(t_np[max_idx], intensity[max_idx], f"Loop {i}",
             fontsize=8, ha="left", va="bottom", color=colors[i])
plt.xlabel("Time")
plt.ylabel("Intensity |A|\u00b2")
plt.title("Strong Pulse Evolution in Ring Resonator")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 HG coefficient comparison (magnitude + phase)

In [ ]:
y_time = A_j_inputs.detach()
A_j_ev_dummy = A_j_final.detach().unsqueeze(-1)
plot_mode_hg_coeffs(
    y_time, A_j_ev_dummy, hg_basis, dt,
    num_modes=num_modes,
    transformation_name=f"Ring Resonator {scenario} (identity)",
)
plt.show()